In [3]:
from game_editor import GameEditor
editor = GameEditor()

In [1]:
from db_manager import DBManager
import os
import json

file_path = r'C:\Users\habib\Documents\GitHub\DataBeach\actions_graded\list_grades_pass_JOMR_nov25_BSD_02.json'

with open(file_path, 'r') as f:
    actions_grades_list = json.load(f)

actions_grades_list
serve_or_pass = 'pass'

with DBManager() as db:
    db.cursor.execute(
        f"""CREATE TABLE IF NOT EXISTS table_{serve_or_pass} (
            action_id TEXT PRIMARY KEY AUTOINCREMENT,
            point_id TEXT,
            paire_id TEXT,
            player TEXT,
            action TEXT,
            grade TEXT,
            point_won BOOLEAN,
            UNIQUE(point_id, player, action)
        )"""
    )


    # for action_graded in actions_grades_list:
    #     db.cursor.execute(
    #         f"""INSERT INTO table_{serve_or_pass} 
    #         (point_id, paire_id, player, action, grade)
    #         VALUES (?, ?, ?, ?, ?)
    #         ON CONFLICT (POINT_ID) DO NOTHING""",
    #         (
    #             action_graded['point_id'],
    #             action_graded['paire_id'],
    #             action_graded['player'],
    #             action_graded['action'],
    #             action_graded['grade']
    #         )
    #     )


✅ Connexion établie : database/databeach_base.db
🔒 Connexion fermée.


In [4]:
import pprint
with DBManager() as db:
    db.execute_query("PRAGMA table_info(table_pass)")
    specs = db.cursor.fetchall()
pprint.pprint(specs)

✅ Connexion établie : database/databeach_base.db
✅ Requête exécutée avec succès.
🔒 Connexion fermée.
[(0, 'pass_id', 'INTEGER', 0, None, 1),
 (1, 'point_id', 'TEXT', 1, None, 0),
 (2, 'paire_id', 'TEXT', 1, None, 0),
 (3, 'player', 'TEXT', 1, None, 0),
 (4, 'action', 'TEXT', 1, None, 0),
 (5, 'grade', 'TEXT', 1, None, 0),
 (6, 'point_won', 'BOOLEAN', 0, None, 0),
 (7, 'created_at', 'TIMESTAMP', 0, 'CURRENT_TIMESTAMP', 0)]


In [36]:
with DBManager() as db:
    table_serve = db.table_to_dataframe('table_serve')
table_serve.head(2)

✅ Connexion établie : database/databeach_base.db
✅ Table 'table_serve' exportée : 144 lignes, 8 colonnes
🔒 Connexion fermée.


,serve_id,point_id,paire_id,player,action,grade,point_won,created_at
0,1,JOMR_jan26_MBV_01_p2,JOMR,RANC Mathilde,serve,out-of-system pass,None,2026-03-14 14:20:18
1,2,JOMR_jan26_MBV_01_p5,JOMR,OFFREDI Jade,serve,error,None,2026-03-14 14:20:18


In [37]:
from db_manager import DBManager

with DBManager() as db:
    table = db.table_to_dataframe('table_point')
table.head(2)

✅ Connexion établie : database/databeach_base.db
✅ Table 'table_point' exportée : 792 lignes, 10 colonnes
🔒 Connexion fermée.


,point_id,service_side,team_a_score,team_b_score,team_a_sets,team_b_sets,team_a_score_diff,team_b_score_diff,point_winner,game_id
0,JOMR_jan26_MBV_01_p1,NoG_AliP,0,0,0,0,0,0,JOMR,JOMR_jan26_MBV_01
1,JOMR_jan26_MBV_01_p2,JOMR,1,0,0,0,1,-1,NoG_AliP,JOMR_jan26_MBV_01


In [3]:
from db_manager import DBManager
import pandas as pd

with DBManager() as db:
    db.execute_query(
        """SELECT table_serve.serve_id, table_serve.point_id
        FROM table_serve
        INNER JOIN table_point ON table_serve.point_id = table_point.point_id
        WHERE table_point.point_winner = ? AND table_serve.grade = ?
        """,
        ("JOMR", "error")
    )
    result = db.cursor.fetchall()

df = pd.DataFrame(result, columns=['serve_id','point_id'])
df

# type(result)

✅ Connection established: c:\Users\habib\Documents\GitHub\DataBeach\database\databeach_base.db
✅ Query executed successfully.
🔒 Connection closed.


,serve_id,point_id
0,315,JOMR_oct25_SSA_05_p19


In [59]:
result

[(14, 'JOMR_jan26_MBV_01_p23')]

In [2]:
# Check les erreurs de grading : faux aces
import pandas as pd
with DBManager() as db:
    db.execute_query(
        """SELECT table_serve.point_id, table_serve.grade
        FROM table_serve
        INNER JOIN table_point ON table_serve.point_id = table_point.point_id
        WHERE table_point.point_winner != ?
        """,
        ("JOMR",)
    )
    result = db.cursor.fetchall()

df = pd.DataFrame(result, columns=['point_id', 'grade'])
errors = df[df['grade']=='ace']
errors

✅ Connexion établie : database/databeach_base.db
✅ Requête exécutée avec succès.
🔒 Connexion fermée.


,point_id,grade


In [1]:
from db_manager import DBManager
with DBManager() as db:
    db.false_aces_corrector(paire_id='JOMR')

✅ Connexion établie : database/databeach_base.db
✅ 1 false aces corrected for 'JOMR'.
🔒 Connexion fermée.
